In [0]:
# %pip install crewai langchain langchain-groq python-dotenv yfinance pandas pandas-ta ddgs agents crewai_tools

In [0]:
import pandas_ta as ta
from ddgs import DDGS
from langchain.tools import tool
from ddgs import DDGS
import yfinance as yf
# ADD THIS LINE FIRST, BEFORE ANY OTHER IMPORTS:
import os
os.environ["OPENAI_API_KEY"] = "sk-dummy-key-for-groq"
from crewai import Agent,Task,Crew
from langchain_groq import ChatGroq

In [0]:
# def get_news(query):
#     with DDGS() as ddgs:
#         results = [
#             r["title"] + ": " + r["body"]
#             for r in ddgs.news(query,max_results=5)
#         ]
#     return results

# def get_fundamentals(symbol):
#     stock = yf.Ticker(symbol)
#     info = stock.info
#     return {
#         "pe": info.get("trailingPE"),
#         "roe": info.get("returnOnEquity"),
#         "debt_to_equity": info.get("debtToEquity"),
#         "market_cap": info.get("marketCap")
#     }

# def get_technical(symbol):
#     df = yf.download(symbol,period="6mo")
#     df["RSI"] = ta.rsi(df["Close"])
#     macd = ta.macd(df["Close"])
#     if macd is not None:
#         df["MACD"] = macd["MACD_12_26_9"]
#     else:
#         df["MACD"] = None
#     return df.tail(1).to_dict()
@tool
def get_news(query: str) -> list:
    """Fetches recent news headlines for a given stock ticker using DuckDuckGo."""
    with DDGS() as ddgs:
        results = [
            r["title"] + ": " + r["body"]
            for r in ddgs.news(query, max_results=5)
        ]
    return results

@tool
def get_fundamentals(symbol: str) -> dict:
    """Fetches key fundamental metrics (PE, ROE, debt/equity, market cap) for a stock symbol."""
    stock = yf.Ticker(symbol)
    info = stock.info
    return {
        "pe": info.get("trailingPE"),
        "roe": info.get("returnOnEquity"),
        "debt_to_equity": info.get("debtToEquity"),
        "market_cap": info.get("marketCap")
    }

@tool
def get_technical(symbol: str) -> dict:
    """Fetches recent technical indicators (RSI, MACD) for a stock symbol."""
    df = yf.download(symbol, period="6mo")
    df["RSI"] = ta.rsi(df["Close"])                                     # works for pandas_ta and ta (momentum.rsi)
    macd = ta.macd(df["Close"])                                         # pandas_ta API – use ta.macd
    if macd is not None:
        df = df.join(macd)                                              # macd is a DataFrame with columns ['MACD_12_26_9', ...]
    last_row = df.tail(1)
    return last_row.to_dict()

# # Optional: collect tools in a list for agents/crew
TOOLS = [get_news, get_fundamentals, get_technical]

In [0]:
# from langchain.tools import tool
# get_news_tool = tool(
#     name="get_news",
#     func=get_news,
#     description="Fetches recent news headlines for a stock"
# )
# get_fundamentals_tool = tool(
#     name="get_fundamentals",
#     func=get_fundamentals,
#     description="Fetches key fundamental metrics for a stock"
# )
# get_technical_tool = tool(
#     name="get_technical",
#     func=get_technical,
#     description="Fetches technical indicators for a stock"
# )

In [0]:
if __name__ == "__main__":
    news = get_news.run("Mobileye Global Inc new past few months")
    fundamentals = get_fundamentals.run("MBLY")
    technical = get_technical.run("MBLY")
    
    print("Latest News:", news)
    print("Fundamentals:", fundamentals)
    print("Technical Indicators:", technical)

In [0]:
llm = ChatGroq(model="llama3-70b-8192", api_key=Groq_API_KEY)
news_agent = Agent(
   role="News Analyst",
   goal="Analyze recent news impacting the stock",
   backstory="Expert financial news analyst",
   llm=llm
)
sentiment_agent = Agent(
   role="Sentiment Analyst",
   goal="Analyze market sentiment from news",
   backstory="Expert in financial sentiment analysis",
   llm=llm
)
fundamental_agent = Agent(
   role="Fundamental Analyst",
   goal="Analyze company financial strength",
   backstory="Equity research analyst",
   llm=llm
)
technical_agent = Agent(
   role="Technical Analyst",
   goal="Analyze price trends and indicators",
   backstory="Chart & indicator specialist",
   llm=llm
)
critic_agent = Agent(
   role="Risk Analyst",
   goal="Identify risks and contradictions",
   backstory="Risk management expert",
   llm=llm
)
decision_agent = Agent(
   role="Investment Strategist",
   goal="Provide final probability-based outlook",
   backstory="Portfolio strategist",
   llm=llm
)

In [0]:
def create_tasks(stock):
    return [
        Task(
            description=f"Analyze recent news for {stock}",
            expected_output="Summary of key news",
            agent=news_agent,
            TOOLS=[get_news]    # Use the TOOL OBJECT, not the function!
        ),
        Task(
            description="Determine sentiment from news",
            expected_output="Positive / Negative sentiment",
            agent=sentiment_agent
        ),
        Task(
            description=f"Analyze fundamentals for {stock}",
            expected_output="Fundamental score and strength",
            agent=fundamental_agent,
            TOOLS=[get_fundamentals]
        ),
        Task(
            description=f"Analyze technical indicators for {stock}",
            expected_output="Trend and momentum",
            agent=technical_agent,
            TOOLS=[get_technical]
        ),
        Task(
            description="Identify risks and conflicting signals",
            expected_output="Risk summary",
            agent=critic_agent
        ),
        Task(
            description="Provide final outlook with probability",
            expected_output="Bullish/Bearish outlook with %",
            agent=decision_agent
        ),
    ]


In [0]:
# def create_tasks(stock):
#     # You must implement task creation logic suitable for your workflow
#     return [
#         Task(agent=news_agent, description=f"Get recent news for {stock}"),
#         Task(agent=fundamental_agent, description=f"Assess fundamentals for {stock}"),
#         Task(agent=technical_agent, description=f"Get technical indicators for {stock}"),
#         Task(agent=sentiment_agent, description=f"Analyze sentiment for {stock}"),
#         Task(agent=critic_agent, description=f"Evaluate risks for {stock}"),
#         Task(agent=decision_agent, description=f"Make strategy decision for {stock}")
#     ]

In [0]:
def run_crew(stock):
    tasks = create_tasks(stock)
    crew = Crew(
        agents=[
            news_agent,
            sentiment_agent,
            fundamental_agent,
            technical_agent,
            critic_agent,
            decision_agent
        ],
        tasks=tasks,
        verbose=True
    )
    result = crew.kickoff()
    return result

In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
if __name__ == "__main__":
    # Test with a known stock symbol, e.g., Apple
    stock_symbol = "NVDA"
    output = run_crew(stock_symbol)
    print("\n=== Crew Analysis Output ===")
    print(output)
